# Fractured Dimension Studio — Free Colab GPU route

Run the image generation and final render on a Google Colab GPU runtime. This notebook uses open-source Diffusers + SDXL + IP-Adapter locally in the Colab session. It does **not** call Cloudflare, Replicate, or any paid image API.

**Limits:** Colab free GPU access is not guaranteed; sessions can disconnect and quotas vary. This is not unlimited. The first run downloads large model weights. Character reference conditioning uses IP-Adapter and may still vary between shots.


In [ ]:
# 1) Clone the authorised project repository
from pathlib import Path
import os
if not Path('/content/fractured-dimension-studio').exists():
    !git clone https://github.com/Raman-Oz9/fractured-dimension-studio.git /content/fractured-dimension-studio
%cd /content/fractured-dimension-studio


In [ ]:
# 2) Install FFmpeg and Python dependencies
!apt-get -qq update && apt-get -qq install -y ffmpeg espeak-ng
!python -m pip install -q --upgrade pip
!python -m pip install -q -e '.[audio]' diffusers transformers accelerate safetensors
import torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU (if offered), then rerun.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 3) Create the sample story (replace this story.json with your own project if desired)
!fractured-dimension create-example --output /content/work/example
from fractured_dimension.project import load_story
story_path = Path('/content/work/example/story.json')
story = load_story(story_path)
print('Story:', story['title'], '| shots:', len(story['shots']))


In [ ]:
# 4) Load SDXL + IP-Adapter. Models are downloaded to this free Colab session.
import torch
from diffusers import StableDiffusionXLPipeline
from PIL import Image
from pathlib import Path
dtype = torch.float16
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=dtype, use_safetensors=True,
)
pipe.load_ip_adapter(
    'h94/IP-Adapter', subfolder='sdxl_models',
    weight_name='ip-adapter_sdxl_vit-h.safetensors',
    image_encoder_folder='models/image_encoder',
)
pipe.set_ip_adapter_scale(0.65)
pipe.enable_model_cpu_offload()
print('SDXL + IP-Adapter ready')


In [ ]:
# 5) Use the repository's own prompt builder, but generate locally on the Colab GPU.
# Reference images are passed to IP-Adapter when available; no hosted image API is used.
import json
import fractured_dimension.images as image_module
from fractured_dimension.images import _shot_prompt, _character_prompt

def local_sdxl_request(prompt, destination, model, reference_images=None):
    refs = reference_images or []
    kwargs = dict(
        prompt=prompt,
        negative_prompt='duplicate head, detached face, extra limbs, malformed hands, distorted anatomy, text, watermark, low resolution, blurry subject',
        width=768, height=1344, num_inference_steps=28, guidance_scale=6.0,
    )
    if refs:
        ref = Image.open(refs[0]).convert('RGB')
        kwargs['ip_adapter_image'] = ref
    result = pipe(**kwargs).images[0]
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    result.save(destination, format='PNG')
    return destination, 'colab-local', 'SDXL-Base-1.0 + IP-Adapter' if refs else 'SDXL-Base-1.0'

image_module._request_image_with_fallback = local_sdxl_request
image_module.REFERENCE_MODELS.add('SDXL-Base-1.0 + IP-Adapter')  # report true when the reference image was actually supplied

def local_character_references(story, root, overwrite, report):
    refs = {}
    for character_id, raw in story.get('character_bible', {}).items():
        profile = raw if isinstance(raw, dict) else {'visual_identity': raw}
        name = profile.get('reference_image')
        if not name:
            continue
        destination = Path(root) / name
        if destination.exists() and not overwrite:
            refs[character_id] = destination
            report.append({'shot': f'character:{character_id}', 'image': str(destination), 'status': 'reference-cached'})
            continue
        prompt = _character_prompt(character_id, profile, story.get('style_lock', ''))
        prompt = (profile.get('reference_prompt', '') + '\\n\\n' + prompt).strip()
        local_sdxl_request(prompt, destination, 'local-sdxl', [])
        refs[character_id] = destination
        report.append({'shot': f'character:{character_id}', 'image': str(destination), 'status': 'reference-generated', 'provider': 'colab-local', 'model': 'SDXL-Base-1.0'})
    return refs

image_module._prepare_character_references = local_character_references
generated = image_module.generate_story_images(story, overwrite=True)
print('Generated images:', len(generated))
print(Path('/content/work/example/generation-report.json').read_text())


In [ ]:
# Free GPU memory before Kokoro narration and FFmpeg rendering
import gc
del pipe
gc.collect()
torch.cuda.empty_cache()
print('Image model unloaded; memory released for narration and rendering.')


In [ ]:
# 6) Render the vertical MP4 with the project's camera paths and Kokoro narration
from fractured_dimension.render import render_story
output = Path('/content/fractured-dimension-short.mp4')
render_story(story, output)
print('Rendered:', output, '| bytes:', output.stat().st_size)


In [ ]:
# 7) Download the finished MP4 to your device
from google.colab import files
files.download('/content/fractured-dimension-short.mp4')
